# DCI vs agentic RAG

Your corpus already exists as pages. The question is what interface the agent gets to it: a retriever that returns ranked chunks, or file tools that list, search, and read the pages themselves. This notebook builds both, holds everything else constant, and scores them on your eval cases.

## Learn | Create | Grow

### Learn
Two ways for an agent to reach documents: read the pages directly through a model-readable wiki, or call a retriever tool. One loop, two tool sets, and what each one reads.


### Create
A wiki index generated from your corpus, both modes scored over your eval cases, and a table of answer quality, calls, evidence read, and latency per mode.


### Grow
Keep the cheaper mode that passes your questions and write down what kind of question would make you switch. Tell your team one question where the modes diverged.


**Estimated time:** 35 minutes
**Reads:** corpus, eval_cases
**Writes:** wiki, agentic_runs

## Setup

Two terms. Direct corpus interaction, DCI, means the agent lists, searches, and reads the pages with file tools. Agentic RAG means the agent calls a retriever that returns ranked chunks and may call it again with a new query. The model, the loop, the questions, and the scoring stay the same; only the tools change.

In [1]:
import json, re, time, textwrap

import pandas as pd
from openai import OpenAI
from rank_bm25 import BM25Okapi

from helpers.config import KEY, LLM_BASE, LLM_MODEL, require, budget
from helpers import workspace as ws, ui
from helpers.llm import client
from helpers.display import show
from helpers.judge import make_judge, parse_json

require("OPENAI_API_KEY")
client = client()
CORPUS_DIR = ws.load_path("corpus")
PAGES = {}
for p in sorted(CORPUS_DIR.rglob("*.md")):
    rel = p.relative_to(CORPUS_DIR).as_posix()
    if not rel.startswith("wiki/") and rel != "vibe_checks.md":   # the wiki is built below; the vibe checks page is the answer key
        PAGES[rel] = p.read_text(encoding="utf-8")
CASES = ws.load("eval_cases")
print(f"✅ model {LLM_MODEL}; {len(PAGES)} pages; {len(CASES)} eval cases")

✅ model gpt-5.5; 24 pages; 5 eval cases


You should see a ✅ line with the model, a page count above five, and at least four eval cases. Stop here if the page count is zero: the corpus has not been built, so run the RAG notebook first or let the seed carry it.

# Learn


## Task 1 of 5 — Build the wiki

An agent with file tools needs a map, or it reads pages at random. The wiki is one markdown index: every page name, what a reader should use it for, and its section headings. The skeleton is built from the headings by hand. The model writes the one-line purpose for each page, from a digest, and you correct it.

In [2]:
def outline(text: str) -> tuple:
    """(title, section headings) of one markdown page."""
    lines = text.splitlines()
    title = next((l[2:].strip() for l in lines if l.startswith("# ")), "")
    return title, [l[3:].strip() for l in lines if l.startswith("## ")]


DIGEST = "\n".join(f"- {name} | {outline(t)[0]} | {' '.join(t.split())[:200]}" for name, t in PAGES.items())
reply = client.chat.completions.create(model=LLM_MODEL, temperature=1, messages=[{"role": "user", "content":
    "For each page below write one line, under fifteen words, saying what a reader should use it for. "
    'Return one JSON object: {"notes": [{"page": "<name>", "use_it_for": "<line>"}]}\n\n' + DIGEST}])
parsed = parse_json(reply.choices[0].message.content) or {}
USE = {n.get("page"): n.get("use_it_for", "") for n in parsed.get("notes", []) if isinstance(n, dict)}

rows = ["# Wiki index", "", "Use this index to decide which page to read. Page names are stable tool inputs.", "",
        "| Page | Use it for | Sections |", "|---|---|---|"]
for name, text in PAGES.items():
    title, heads = outline(text)
    rows.append(f"| `{name}` | {USE.get(name) or title or name} | {', '.join(heads[:5])} |")
rows += ["", "Suggested navigation:", "",
         "- Start from the charter for what the product is and refuses to do, then follow a term to the page that operates on it.",
         "- For a question about a past request, read the whole transcript rather than one matching line.",
         "- When a search matches lines on several pages, read each page's section before answering."]
WIKI = "\n".join(rows)
ws.save("wiki", WIKI)
show(WIKI)

✅ wrote wiki → workspace/corpus/wiki/index.md (36 lines)


# Wiki index

Use this index to decide which page to read. Page names are stable tool inputs.

| Page | Use it for | Sections |
|---|---|---|
| `charter.md` | Understand Deskmate’s problem, users, scope, constraints, and success criteria. | The problem, The users, The product vision, What a good answer looks like, Where it will be wrong |
| `prompts/context-charter.md` | See how charter context enables specific, grounded answers. | Example 1 |
| `prompts/context-none.md` | See how missing context forces uncertainty and refusal to invent details. | Example 1 |
| `prompts/few-shot-two-examples.md` | Learn classification behavior from two labeled helpdesk examples. | Example 1 |
| `prompts/few-shot-zero.md` | Compare zero-shot classification without examples. | Example 1 |
| `prompts/meta-prompt-applied.md` | Review outputs from an applied generated prompt. | Example 1 |
| `prompts/meta-prompt-generate.md` | Generate a reusable prompt from the Deskmate charter. | Example 1, The problem, The users, The product vision, What a good answer looks like |
| `prompts/persona-none.md` | Compare neutral tone without a specified persona. | Example 1 |
| `prompts/persona-patient.md` | Model a patient, reassuring helpdesk response style. | Example 1 |
| `prompts/persona-terse.md` | Model a concise, direct helpdesk response style. | Example 1 |
| `prompts/reasoning-high.md` | Examine answers using more explicit reasoning. | Example 1 |
| `prompts/reasoning-none.md` | Compare answers with minimal stated reasoning. | Example 1 |
| `prompts/self-refine-draft.md` | Use as the initial product pitch draft. | Example 1 |
| `prompts/self-refine-revised.md` | Use as the refined product pitch after critique. | Example 1, REVISED |
| `prompts/stacked.md` | Study combined prompting for structured product decisions. | Example 1 |
| `prompts/structured-output.md` | Learn enforcing JSON schemas for charter-based extraction. | Example 1, The problem, The users, The product vision, What a good answer looks like |
| `transcripts/t01.md` | Review VPN staging-database answer that escalates due missing verified details. | Tool calls |
| `transcripts/t02.md` | Review analytics warehouse access guidance with escalation for unknown entitlement details. | Tool calls |
| `transcripts/t03.md` | Review refusal to approve access while logging escalation. | Tool calls |
| `transcripts/t04.md` | Review out-of-scope vacation-days handling and charter search. | Tool calls |
| `transcripts/t05.md` | Compare VPN answer that suggests split tunneling but still escalates. | Tool calls |
| `transcripts/t06.md` | Compare analytics access answer emphasizing unverified sources and ticket creation. | Tool calls |
| `transcripts/t07.md` | Compare access-approval refusal grounded in entitlement safety limits. | Tool calls |
| `transcripts/t08.md` | Compare concise out-of-scope PTO response and charter search. | Tool calls |

Suggested navigation:

- Start from the charter for what the product is and refuses to do, then follow a term to the page that operates on it.
- For a question about a past request, read the whole transcript rather than one matching line.
- When a search matches lines on several pages, read each page's section before answering.

You should see a ✅ line and a rendered table with one row per page, a purpose line, and its sections. Stop here if the purpose column repeats the page title for every row: the model did not return JSON, so print `reply.choices[0].message.content` and check it.

## Task 2 of 5 — Two corpus interfaces

Agentic RAG gets one tool. `search_chunks` returns the top sections by BM25 and can be called again with a new query, but it cannot list pages or ask for a whole one. DCI gets three: `list_pages` returns the wiki, `grep_wiki` returns matching lines with the page and line number, `read_page` returns a whole page. In DCI the agent, not a retriever, decides what to read next.

In [3]:
TOKEN = re.compile(r"[a-z0-9$%]+")


def tokenize(text: str) -> list:
    return TOKEN.findall(text.lower())


chunks = []
for page, text in PAGES.items():
    sections = re.split(r"(?=^## )", text, flags=re.M)
    for i, section in enumerate(s for s in sections if s.strip()):
        chunks.append({"id": f"{page}#s{i}", "page": page, "text": section.strip()})
bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])


def search_chunks(query: str, k: int = 4) -> str:
    scores = bm25.get_scores(tokenize(query))
    order = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:max(1, min(k, 8))]
    return "\n\n---\n\n".join(f"[{chunks[i]['id']}]\n{chunks[i]['text'][:1000]}" for i in order)


def list_pages() -> str:
    return WIKI


def grep_wiki(pattern: str) -> str:
    try:
        rx = re.compile(pattern, re.I)
    except re.error:
        rx = re.compile(re.escape(pattern), re.I)
    hits = []
    for page, text in PAGES.items():
        for number, line in enumerate(text.splitlines(), 1):
            if rx.search(line):
                hits.append(f"{page}:{number}: {line}")
    return "\n".join(hits[:40]) or "(no matches)"


def read_page(page: str) -> str:
    return PAGES[page][:12000] if page in PAGES else f"(unknown page: {page})"


def schema(name, description, properties, required=()):
    return {"type": "function", "function": {"name": name, "description": description,
            "parameters": {"type": "object", "properties": properties, "required": list(required)}}}


RAG_FUNCS = {"search_chunks": search_chunks}
RAG_TOOLS = [schema("search_chunks", "Search ranked corpus sections. Search again with a new query when evidence is incomplete.",
                    {"query": {"type": "string"}, "k": {"type": "integer"}}, ["query"])]
DCI_FUNCS = {"list_pages": list_pages, "grep_wiki": grep_wiki, "read_page": read_page}
DCI_TOOLS = [
    schema("list_pages", "Read the wiki index: every page name, what it is for, and its sections.", {}),
    schema("grep_wiki", "Search exact text or a regular expression across all pages; returns page:line: text.",
           {"pattern": {"type": "string"}}, ["pattern"]),
    schema("read_page", "Read one complete page by its name from the index.",
           {"page": {"type": "string", "enum": sorted(PAGES)}}, ["page"]),
]
MODES = {"agentic_rag": (RAG_TOOLS, RAG_FUNCS), "dci": (DCI_TOOLS, DCI_FUNCS)}

print(f"RAG index: {len(chunks)} sections over {len(PAGES)} pages")
print(textwrap.shorten(search_chunks(CASES[0]["question"], 2), 300))
print(grep_wiki("ticket")[:300])

RAG index: 63 sections over 24 pages
[transcripts/t05.md#s1] ## Tool calls - `search_prompt_outputs` {"query": "VPN connects but cannot reach staging database split tunnel menu path"} → - `search_charter` {"query": "VPN connects cannot reach staging database split tunnel setting exact menu path"} → [What a good answer looks like] [...]
charter.md:5: Every engineer in the building files a helpdesk ticket about once a month:
charter.md:14:   from the VPN" at 9 am and wants a fix, not a ticket number.
charter.md:21: ticket history, opens a ticket when it cannot, and never resets anything
charter.md:25: user's ticket text.
charter.md:


You should see the section count, a chunk result with a `page#s` id, and grep lines in `page:line:` form. Stop here if grep returns no matches for a word you know is in the corpus: the pattern is being compiled as a regex, so escape it.

### ❓ Question
Which of the four tools could leak one user's transcript into another user's answer, and what would you check on the caller before running it?

Answer:

## Task 3 of 5 — One loop for both

The loop is the same for both modes: send the question and the tools, run whatever the model calls, append the results, repeat until it answers or hits the turn limit. It records every call and how many characters came back. Keeping the loop identical means any difference in behaviour comes from the interface, not the orchestration.

In [4]:
SYSTEM = ("Answer questions about the product only from tool evidence. Investigate before answering. "
          "Name the page you used. If the evidence is incomplete, say so.")


def run_agent(question: str, tools: list, funcs: dict, max_turns: int = 6) -> dict:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    trace, chars, started = [], 0, time.perf_counter()
    for _ in range(max_turns):
        msg = client.chat.completions.create(model=LLM_MODEL, temperature=1, messages=messages,
                                             tools=tools, tool_choice="auto").choices[0].message
        calls = msg.tool_calls or []
        if not calls:
            return {"answer": (msg.content or "").strip(), "trace": trace, "evidence_chars": chars,
                    "latency_s": round(time.perf_counter() - started, 2), "stopped": "answer"}
        shaped = [{"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
                  for c in calls]
        messages.append({"role": "assistant", "content": msg.content or "", "tool_calls": shaped})
        for call in shaped:
            name = call["function"]["name"]
            try:
                args = json.loads(call["function"]["arguments"] or "{}")
                if name not in funcs:
                    raise KeyError(f"unknown tool {name}")
                result = str(funcs[name](**args))
            except Exception as exc:
                args, result = {}, f"(tool error: {type(exc).__name__}: {exc})"
            chars += len(result)
            trace.append({"tool": name, "args": args, "chars": len(result)})
            messages.append({"role": "tool", "tool_call_id": call["id"], "content": result})
    return {"answer": "(max turns reached)", "trace": trace, "evidence_chars": chars,
            "latency_s": round(time.perf_counter() - started, 2), "stopped": "max_turns"}


question = CASES[0]["question"]
print("Q:", question)
for mode, (tools, funcs) in MODES.items():
    r = run_agent(question, tools, funcs)
    print(f"\n[{mode}] {' -> '.join(t['tool'] for t in r['trace']) or '(no tools)'} | {r['evidence_chars']} chars | {r['latency_s']}s")
    print(textwrap.shorten(r["answer"], 400))

Q: My VPN connects but I cannot reach staging.



[agentic_rag] search_chunks -> search_chunks | 8085 chars | 10.76s
Using **transcripts/t05.md** and **charter.md**: This is likely related to the VPN **split-tunnel** setting. However, the available evidence does **not** include the verified menu path or exact fix steps, so I shouldn’t invent them. Please collect: - OS and VPN client/version - The staging hostname/URL you’re trying to reach - Exact error message - Whether other staging/internal resources [...]



[dci] list_pages -> read_page -> read_page -> read_page | 7788 chars | 6.56s
This is likely the VPN **split-tunnel setting**, but I don’t have the verified exact menu path or fix steps in the available product evidence. I’m basing this on `charter.md`, which says a good answer should name the split tunnel setting and exact menu path, and on `transcripts/t01.md` / `transcripts/t05.md`, where the system escalated because those verified details were missing. Please open [...]


You should see one block per mode: the tool sequence, the characters of evidence read, the latency, and a short answer that names a page. Stop here if a mode says max turns reached: the model is looping on the same call, so read its trace before scoring anything.

# Create


## Task 4 of 5 — Score every case

Run every eval case through both modes. Your judge scores each answer against the reference from the eval case, 0 to 10. If the case names the pages that hold the evidence, the record also says whether the answer named one of them. Calls, evidence characters, and latency go in the same row, so a correct answer cannot hide a wasteful route.

In [5]:
coverage = make_judge("coverage",
    "Question: {question}\nA good answer includes: {reference}\nAnswer: {response}\n\n"
    "Score 0-10 how completely and accurately the answer covers what a good answer includes.",
    needs_reference=True)

RUNS = []
for case in ui.track(CASES[:budget(len(CASES), 3)], "both modes"):
    for mode, (tools, funcs) in MODES.items():
        r = run_agent(case["question"], tools, funcs)
        verdict = coverage({"question": case["question"], "response": r["answer"], "reference": case["reference"]})
        expected = case.get("pages") or []
        RUNS.append({"question": case["question"], "mode": mode, "answer": r["answer"], "case_id": case["id"],
                     "score": verdict["score"], "rationale": verdict["rationale"],
                     "named_expected_page": any(p.lower() in r["answer"].lower() for p in expected) if expected else None,
                     "calls": len(r["trace"]), "evidence_chars": r["evidence_chars"], "latency_s": r["latency_s"],
                     "stopped": r["stopped"], "trace": r["trace"]})
ws.save("agentic_runs", RUNS)
results = pd.DataFrame(RUNS)
ui.table(results.pivot(index="case_id", columns="mode", values="score"), title="judge score per case", float_fmt="{:.0f}")

✅ wrote agentic_runs → workspace/retrieval/agentic_runs.jsonl (10 rows)


     judge score per case      
┏━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━┓
┃ case_id ┃ agentic_rag ┃ dci ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━┩
│ v01     │           4 │   6 │
│ v02     │           4 │   2 │
│ v03     │          10 │  10 │
│ v04     │          10 │   8 │
│ v05     │           0 │   0 │
└─────────┴─────────────┴─────┘

You should see a ✅ line with two rows per case and a table of judge scores, one column per mode. Stop here if a column is all None: the judge did not return JSON, so print one rationale and check the model's reply.

### ❓ Question
Find the case with the biggest score gap between modes. Did the losing mode retrieve the wrong evidence, or retrieve the right evidence and answer badly?

Answer:

## Task 5 of 5 — Inspect the difference

There is no universal winner. Agentic RAG is usually cheaper when a question maps to one or two sections. DCI earns its extra calls when page structure, exact identifiers, or evidence on two pages matter, and it gives the model broader raw access, which production must gate per caller. Read the traces, not only the averages.

In [6]:
summary = results.groupby("mode").agg(score=("score", "mean"), calls=("calls", "mean"),
                                      evidence_chars=("evidence_chars", "mean"), latency_s=("latency_s", "mean"))
ui.table(summary, title="average per mode", float_fmt="{:.2f}")

for case_id, group in results.groupby("case_id", sort=False):
    print(f"\n=== {case_id}: {group.iloc[0]['question']}")
    for _, row in group.iterrows():
        route = " -> ".join(t["tool"] for t in row["trace"]) or "(no tools)"
        print(f"[{row['mode']}] score {row['score']} | {route} | {row['evidence_chars']} chars")
        print("   ", textwrap.shorten(row["answer"], 260))

                      average per mode                      
┏━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ mode        ┃ score ┃ calls ┃ evidence_chars ┃ latency_s ┃
┡━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ agentic_rag │  5.60 │  2.40 │       10898.00 │      8.07 │
│ dci         │  5.20 │  2.60 │        7945.60 │      7.22 │
└─────────────┴───────┴───────┴────────────────┴───────────┘


=== v01: My VPN connects but I cannot reach staging.
[agentic_rag] score 4 | search_chunks -> search_chunks -> search_chunks -> search_chunks | 18739 chars
    Using **transcripts/t05.md** and **prompts/meta-prompt-applied.md**: This is likely related to the VPN **split-tunnel setting**, but the available evidence does **not** include the verified menu path or exact fix steps, so I don’t want to invent them. [...]
[dci] score 6 | list_pages -> read_page -> read_page -> read_page | 7788 chars
    This is likely the VPN **split tunnel** setting. However, the available Deskmate evidence does **not** include the verified VPN client menu path or exact fix steps. The **Charter: Deskmate** says a good answer should name the split tunnel setting and [...]

=== v02: How do I get access to the analytics warehouse?
[agentic_rag] score 4 | search_chunks -> search_chunks -> search_chunks -> search_chunks | 18727 chars
    I don’t have verified instructions for getting analytics warehouse access. T

You should see a two-row summary and, per case, both traces with their scores and answers. Stop here if DCI never calls `read_page`: it is answering from grep lines alone, so tighten the system prompt to require reading the page before answering.

## Your turn

Add two questions from your own product: one that needs an exact string from a page, and one that needs evidence from two pages. Name the pages before you run either mode. Then run both modes and say which interface you would ship for each question and why.

In [7]:
MY_CASES = [
    {"id": "exact", "question": "", "reference": "", "pages": []},
    {"id": "two_pages", "question": "", "reference": "", "pages": []},
]
for case in [c for c in MY_CASES if c["question"]]:
    print(f"\n=== {case['id']}: {case['question']}")
    for mode, (tools, funcs) in MODES.items():
        r = run_agent(case["question"], tools, funcs)
        named = [p for p in case["pages"] if p.lower() in r["answer"].lower()]
        print(f"[{mode}] {' -> '.join(t['tool'] for t in r['trace']) or '(no tools)'} | named {named or 'no expected page'}")
        print("   ", textwrap.shorten(r["answer"], 300))
if not any(c["question"] for c in MY_CASES):
    print("fill in at least one case above, then rerun")

fill in at least one case above, then rerun


# Grow


## From prototype to production

| What we built | Production equivalent |
|---|---|
| BM25 over markdown sections | Dense or hybrid retrieval with a reranker and index freshness |
| Every page visible to every call | Per-user authorisation on list, search, and read |
| A wiki index written once | Generated navigation with link checks and page ownership |
| One judge against a reference line | Human-calibrated scoring of answers and citations |
| In-process tools and a turn limit | Timeouts, rate limits, tracing, and durable run records |
| Two modes compared on a handful of cases | A routing decision reviewed as the corpus changes |

## Responsible controls

- File tools scoped to the corpus directory and read-only.
- Evidence read per answer logged so cost is visible per mode.
- A rule for which mode handles which question type, reviewed as the corpus grows.


## Grow further

- Swap `search_chunks` for the best rung of your retrieval ladder and rerun the comparison; note which cases change sides.
- Add a caller id to every DCI tool and refuse `read_page` on a transcript that belongs to another user.
- Write a router: send a question to DCI only when the wiki index names a page for one of its terms, otherwise to agentic RAG, and compare cost per case.